In [ ]:
pip install --upgrade google-cloud-bigquery

Set environment variables for GCP connection

In [ ]:
export GOOGLE_APPLICATION_CREDENTIALS="path/to/service_account.json"

Creating connection to BigQuery Project

In [1]:
from google.cloud import bigquery

def query_bigquery_to_df(query: str) -> 'pd.DataFrame':
    """Executes a BigQuery SQL query and returns the results as a pandas DataFrame.

    Args:
        query (str): The SQL query string to execute.

    Returns:
        pd.DataFrame: DataFrame containing query results.
    """
    client = bigquery.Client()
    query_job = client.query(query)
    df = query_job.result().to_dataframe()
    return df



sample_query = """
    SELECT *
    FROM `project_id.dataset.table`
    LIMIT 100
    """

    df = query_bigquery_to_df(sample_query)
    print(df.head())

ModuleNotFoundError: No module named 'google'

Initial Data Understanding and Exploration Template
- Shape of Dataframe
- Preview min and max records
- Column names
- Data Types
- Schema Validation
- Missing Data
- Basic Statistics
- Duplicate and Uniqueness Checks

In [ ]:
def summarize_dataframe(df):
    """
    Prints out basic information about a DataFrame:
    1. Shape (number of rows and columns)
    2. First 10 rows (head)
    3. Last 10 rows (tail)
    4. Column names
    5. Data types
    """
    # Shape of DataFrame
    rows, cols = df.shape
    print(f"Rows: {rows}, Columns: {cols}\n")
    
    # Preview of the first 10 records
    print("First 10 records:")
    display(df.head(10))
    
    # Preview of the last 10 records
    print("Last 10 records:")
    display(df.tail(10))
    
    # Column Names
    print("Column Names:")
    print(df.columns)
    print()
    
    # Data Types
    print("Data Types:")
    print(df.dtypes)

summarize_dataframe(df)

Data Checks

In [ ]:
def data_checks(df):
    """
    Perform the following checks on a DataFrame:
    1. Show descriptive statistics (for numeric columns)
    2. Show descriptive statistics for object (categorical) columns
    3. Check for duplicated rows
    """
    
    # 1. Descriptive statistics (numeric columns)
    print("Descriptive statistics (numeric columns):")
    display(df.describe())
    
    # 2. Descriptive statistics (categorical/object columns)
    print("Descriptive statistics (categorical columns):")
    display(df.select_dtypes(include='object').describe())
    
    # 3. Duplicates
    duplicate_rows = df[df.duplicated()]
    print(f"Number of duplicate rows: {len(duplicate_rows)}")


Detecting Outlier in the Data

In [ ]:
import pandas as pd

def detect_outliers(df):
    """
    Detect outliers in each numeric column of the given DataFrame
    using the 1.5 * IQR rule.
    
    Prints the number of outliers for each column and optionally displays 
    the outlier rows.
    """
    # Identify numeric columns (integers and floats)
    numeric_cols = df.select_dtypes(include=['number']).columns

    for col in numeric_cols:
        # Compute Q1, Q3, and IQR
        Q1 = df[col].quantile(0.25)
        Q3 = df[col].quantile(0.75)
        IQR = Q3 - Q1
        
        # Determine lower and upper bounds
        lower_bound = Q1 - 1.5 * IQR
        upper_bound = Q3 + 1.5 * IQR
        
        # Boolean mask for outliers
        outlier_mask = (df[col] < lower_bound) | (df[col] > upper_bound)
        outliers = df[outlier_mask]
        
        print(f"Column '{col}': {len(outliers)} outliers found.")
        
        # Uncomment the next line if you want to display the actual outlier rows:
        display(outliers)
        
detect_outliers(df)


Correlation and Relationships between features and labels

In [ ]:
correlation_matrix = df.corr(numeric_only=True)
display(correlation_matrix)

Plotting of all features and columns based on data types and conditions
- Historgrams
- Density Plots
- Boxplot
- Scatter Plot

In [3]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

def plot_all_columns(df):
    """
    Loop over all columns in the DataFrame and generate plots based on data type:
      - Numeric columns: Histogram, Density Plot, Boxplot, and Scatter Plot vs. index.
      - Categorical columns: Bar Plot of value counts.
    """
    for col in df.columns:
        print(f"\nPlotting data for column: {col}")
        
        # Check if the column is numeric
        if pd.api.types.is_numeric_dtype(df[col]):
            # Create a 2x2 grid of subplots for numeric data
            fig, axes = plt.subplots(2, 2, figsize=(12, 10))
            
            # Histogram
            axes[0, 0].hist(df[col].dropna(), bins=30, edgecolor='k')
            axes[0, 0].set_title(f'Histogram of {col}')
            
            # Density Plot using seaborn
            sns.kdeplot(df[col].dropna(), ax=axes[0, 1])
            axes[0, 1].set_title(f'Density Plot of {col}')
            
            # Box Plot
            axes[1, 0].boxplot(df[col].dropna())
            axes[1, 0].set_title(f'Boxplot of {col}')
            
            # Scatter Plot of values vs. DataFrame index
            axes[1, 1].scatter(df.index, df[col], alpha=0.5)
            axes[1, 1].set_title(f'Scatter Plot of {col} vs. Index')
            axes[1, 1].set_xlabel('Index')
            axes[1, 1].set_ylabel(col)
            
            plt.tight_layout()
            plt.show()
        
        # Check if the column is categorical
        elif pd.api.types.is_object_dtype(df[col]) or pd.api.types.is_categorical_dtype(df[col]):
            # Calculate value counts and create a bar plot
            value_counts = df[col].value_counts()
            plt.figure(figsize=(8, 4))
            plt.bar(value_counts.index.astype(str), value_counts.values)
            plt.title(f'Bar Plot of {col}')
            plt.xlabel(col)
            plt.ylabel('Count')
            plt.xticks(rotation=45)
            plt.show()
        
        else:
            print(f"Column '{col}' is not recognized as numeric or categorical; skipping plotting.")

# Example usage:
plot_all_columns(df)


SyntaxError: invalid syntax (2590094549.py, line 1)

Handling Missing Data, solutions:
- Remove rows with missing data
- Impute missing values with mean values
- Replace missing values with zero

In [ ]:
import pandas as pd

def remove_missing_rows(df):
    """
    Remove rows that contain any missing values.
    
    Parameters:
        df (pd.DataFrame): The input DataFrame.
    
    Returns:
        pd.DataFrame: A DataFrame with all rows containing missing values removed.
    """
    return df.dropna()

def impute_missing_with_average(df, column):
    """
    Impute missing values in a specific column with the average (mean) of that column.
    
    Parameters:
        df (pd.DataFrame): The input DataFrame.
        column (str): The name of the column in which to impute missing values.
    
    Returns:
        pd.DataFrame: A new DataFrame where missing values in the specified column are replaced by the column's mean.
    """
    df_copy = df.copy()
    mean_value = df_copy[column].mean()
    df_copy[column].fillna(mean_value, inplace=True)
    return df_copy

def set_missing_to_zero(df):
    """
    Replace all missing values in the DataFrame with zero.
    
    Parameters:
        df (pd.DataFrame): The input DataFrame.
    
    Returns:
        pd.DataFrame: A DataFrame where every missing value has been set to zero.
    """
    return df.fillna(0)

# Choose desired missing data handling solution
cleaned_df = remove_missing_rows(df)
imputed_df = impute_missing_with_average(df, 'column_with_missing_values')
zero_imputed_df = set_missing_to_zero(df)


Analyze and Convert Text Data
- Cleaning text data
- - lowercasing, removing punctuation, removing extra whitespace
- Calculating word frequencies
- Vectorizing text

In [ ]:
import pandas as pd
import re
import string
from collections import Counter
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer

def clean_text(text):
    """
    Clean a text string by converting it to lowercase, removing punctuation, and extra whitespace.
    
    Parameters:
        text (str): The input text string.
        
    Returns:
        str: The cleaned text.
    """
    # Convert to lowercase
    text = text.lower()
    # Remove punctuation using regex
    text = re.sub(f"[{re.escape(string.punctuation)}]", "", text)
    # Remove extra spaces and newlines
    text = re.sub(r"\s+", " ", text).strip()
    return text

def apply_text_cleaning(df, column):
    """
    Apply text cleaning to a specified column in the DataFrame.
    
    Parameters:
        df (pd.DataFrame): The input DataFrame.
        column (str): The name of the column containing text data.
    
    Returns:
        pd.DataFrame: A new DataFrame with the cleaned text in the specified column.
    """
    df_copy = df.copy()
    # Ensure the column is treated as strings then apply cleaning
    df_copy[column] = df_copy[column].astype(str).apply(clean_text)
    return df_copy

def word_frequency(df, column):
    """
    Compute the word frequency for text data in a specified DataFrame column.
    
    Parameters:
        df (pd.DataFrame): The input DataFrame.
        column (str): The name of the column containing text data.
    
    Returns:
        dict: A dictionary mapping each word to its frequency count.
    """
    # Combine all text entries into a single string and clean it
    all_text = " ".join(df[column].dropna().astype(str))
    cleaned_text = clean_text(all_text)
    # Split text into words and count frequencies
    words = cleaned_text.split()
    freq_dict = Counter(words)
    return dict(freq_dict)

def vectorize_text(df, column, method='count'):
    """
    Convert text data into numerical features using vectorization techniques.
    
    Parameters:
        df (pd.DataFrame): The input DataFrame.
        column (str): The name of the column containing text data.
        method (str): The vectorization method to use. Options are 'count' for CountVectorizer 
                      or 'tfidf' for TfidfVectorizer.
    
    Returns:
        tuple: A tuple (vectorizer, feature_matrix) where 'vectorizer' is the fitted vectorizer 
               object and 'feature_matrix' is the resulting sparse matrix of features.
    """
    # Create a copy and clean the text column
    df_copy = df.copy()
    df_copy[column] = df_copy[column].astype(str).apply(clean_text)
    
    if method == 'count':
        vectorizer = CountVectorizer()
    elif method == 'tfidf':
        vectorizer = TfidfVectorizer()
    else:
        raise ValueError("method must be either 'count' or 'tfidf'")
    
    feature_matrix = vectorizer.fit_transform(df_copy[column])
    return vectorizer, feature_matrix

# Example usage:
if __name__ == "__main__":
    
    # Clean the text data in the 'review' column
    cleaned_df = apply_text_cleaning(df, 'review')
    print("Cleaned DataFrame:")
    print(cleaned_df)
    
    # Compute word frequency in the 'review' column
    freq = word_frequency(df, 'review')
    print("\nWord Frequency:")
    print(freq)
    
    # Vectorize text using CountVectorizer
    vectorizer, features = vectorize_text(df, 'review', method='count')
    print("\nFeature Matrix Shape (Count Vectorizer):", features.shape)


Tokenization and Visualizing Text Data